In [ ]:
# Dependencies

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler


from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Load Train Data

train_data = pd.read_csv('training_data.csv', sep='delimiter', header=None)
print(train_data.shape)
train_data.head(2)

In [ ]:
# Data Preparation
# Transforming into tabular data

train_data= train_data[0].str.split(';', n=160, expand=True)
train_data, train_data.columns = train_data[1:] , train_data.iloc[0]
train_data.head(2)

In [ ]:
# Replacing , with .

train_data = pd.DataFrame({col: train_data[col].str.replace(',', '.') for col in train_data.columns})
train_data.head(2)

In [ ]:
# Handle Missing Values Differently
train_data.replace({'NA': None}, inplace=True)
train_data.fillna(value=np.nan, inplace=True)  # Convert empty strings to NaN

# Calculate the percentage of missing data for each column
missing_percentages = train_data.isnull().mean() * 100
print(missing_percentages)

# Define a threshold for dropping columns
threshold = 5  # Dropping columns with more than 5% missing values
columns_to_drop = missing_percentages[missing_percentages > threshold].index
print(columns_to_drop)

# Drop these columns from the dataset
train_data.drop(columns=columns_to_drop, inplace=True)

In [ ]:
# Convert the Group column to a one hot encoded Data Frame

display(train_data['Group'].value_counts())
train_data = pd.get_dummies(train_data, columns=['Group'], drop_first=True, prefix='G')
train_data = train_data.replace({False: 0, True: 1})

# Print the columns names

print(train_data.columns)
train_data.head(2)

In [ ]:
train_data = train_data.drop(['Perform'], axis=1)
train_data.shape

In [ ]:
for col in train_data.columns:
    train_data[col] = pd.to_numeric(train_data[col], errors='coerce')

In [ ]:
train_data.dtypes

In [ ]:
train_data.describe()

In [ ]:
# Loop over each column, except the last 10
for col in train_data.columns[:-10]:
    q1 = train_data[col].quantile(0.25)
    q3 = train_data[col].quantile(0.75)
    iqr = q3 - q1
    fence_low = q1 - 1.5 * iqr
    fence_high = q3 + 1.7 * iqr
    
    # Filter out outliers
    filtered_data = train_data.loc[(train_data[col] >= fence_low) & (train_data[col] <= fence_high)].copy()

# Compute the correlation with the 'Class' column
correlation_matrix = filtered_data.corrwith(filtered_data["Class"]).sort_values(ascending=False).reset_index()

# Print the correlated features
print(correlation_matrix)

In [ ]:
# corr_features = correlation_matrix[1:55]['index'].tolist() + correlation_matrix[-45:]['index'].tolist()
corr_features = correlation_matrix[1:45]['index'].tolist() + correlation_matrix[-35:]['index'].tolist()

In [ ]:
display(train_data['Class'].value_counts())

In [ ]:
print(train_data.shape)
train_data = train_data[train_data.isna().sum(axis=1) < 2]
print(train_data.shape)

In [ ]:
display(train_data['Class'].value_counts())

In [ ]:
# col_with_no_missing_vals = []
# col_with_minor_missing_vals = []
# col_with_major_missing_vals = []

# # summarize the number of rows with missing values for each column
# for i in range(train_data.shape[1]):
#     # count number of rows with missing values
#     n_miss = train_data.iloc[:,i].isna().sum()
#     perc = n_miss / train_data.shape[0] * 100
#     if perc >=0.5:
#         col_with_major_missing_vals.append(train_data.columns[i])
#     elif perc <0.5 and perc >0.00:
#         col_with_minor_missing_vals.append(train_data.columns[i])
#     elif perc >= 0.00:
#         col_with_no_missing_vals.append(train_data.columns[i])
# print(f'{len(col_with_no_missing_vals)} out of {len(train_data.columns)} have no missing values')
# print(f'{len(col_with_minor_missing_vals)} out of {len(train_data.columns)} have missing values < 1%')
# print(f'{len(col_with_major_missing_vals)} out of {len(train_data.columns)} have missing values > 1%')

In [ ]:
# print(train_data.shape)
# train_data = train_data.dropna(subset=col_with_major_missing_vals)
# print(train_data.shape)

In [ ]:
display(train_data['Class'].value_counts())

In [ ]:
X = train_data[corr_features]
y = train_data['Class'].astype('int')
print(X.shape, y.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.80, random_state=42, stratify=y)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

In [ ]:
imputer = KNNImputer(n_neighbors=3, weights="uniform")
# transform the dataset
X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)

In [ ]:
# # Initialize SMOTE 

# smote = SMOTE(random_state=42, k_neighbors = 5)
# X_bal, y_bal = smote.fit_resample(X_train, y_train)

# # fit predictor and target variable
# X_bal, y_bal = smote.fit_resample(X_train, y_train)


smote_tomek = SMOTETomek(smote=SMOTE(random_state=42, k_neighbors=5), 
                         tomek=TomekLinks(sampling_strategy='auto'))

# Apply SMOTETomek to the data
X_bal, y_bal = smote_tomek.fit_resample(X_train, y_train)

display(y_train.value_counts())
display(y_bal.value_counts())

In [ ]:
# Pipeline
pipeline = Pipeline([
    # ('scaler', StandardScaler()),  # Normalize data
    ('classifier', GradientBoostingClassifier())
])

# Grid of parameters to choose from
parameters = {
    'classifier__n_estimators': [150, 200],
    'classifier__learning_rate': [0.01, 0.1],
    'classifier__max_depth': [3, 5, 9, 20],
    'classifier__subsample': [0.8, 1.0]
}

In [ ]:
# Model
# model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=20)
# model.fit(X_bal, y_bal)

# Setup the grid search
model = GridSearchCV(pipeline, parameters, cv=5, scoring='accuracy', verbose=1)
model.fit(X_bal, y_bal)

In [ ]:
# Create predictions on X_test
predictions = model.predict(X_val)
print(predictions[0:10])

print(model.score(X_val, y_val))

In [ ]:
print(classification_report(y_val, predictions))

In [ ]:
cm = confusion_matrix(predictions, y_val)
cm

In [ ]:
cost_matrix = [[0,1,2], [1,0,1], [2,1,0]]
cost_matrix

In [ ]:
err = np.sum((cm*cost_matrix)/len(y_val))
print(err)

In [ ]:
test_data = pd.read_csv('test_data_no_target.csv', sep='delimiter', header=None)
print(test_data.shape)
test_data.head(2)

In [ ]:
test_data= test_data[0].str.split(';', n=160, expand=True)
test_data, test_data.columns = test_data[1:] , test_data.iloc[0]
test_data.head(2)

In [ ]:
test_data = pd.DataFrame({col: test_data[col].str.replace(',', '.') for col in test_data.columns})
test_data.head(2)

In [ ]:
test_data = pd.get_dummies(test_data, columns=['Group'], drop_first=True, prefix='G')
test_data = test_data.replace({False: 0, True: 1})

In [ ]:
# Replace 'NA' and empty strings with NaN
test_data.replace(['NA', ''], np.nan, inplace=True)
test_data.drop(columns=columns_to_drop, inplace=True)

In [ ]:
X_test = test_data[corr_features]
X_test = imputer.transform(X_test)
test_predictions = model.predict(X_test)
print(test_predictions[:5])
print(test_predictions.shape)

In [ ]:
pd.DataFrame(test_predictions).to_csv('prediction.txt', index=False, header=False)